# EDA - Zalo AI Traffic Sign Dataset
Notebook này được dùng để phân tích dữ liệu, vẽ biểu đồ chứng minh mất cân bằng class và chứng minh bài toán nhận diện vật thể siêu nhỏ (Small Object Detection).

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import glob

# Tự động dò tìm đường dẫn file JSON trên Kaggle để không bao giờ bị lỗi sai đường dẫn nữa
json_paths = glob.glob('/kaggle/input/**/train_traffic_sign_dataset.json', recursive=True)
if not json_paths:
    raise FileNotFoundError("Không tìm thấy file JSON trong /kaggle/input/. Bạn đã Add Dataset vào notebook chưa?")

json_file_path = json_paths[0]
print(f"Đã tìm thấy file JSON tại: {json_file_path}")

with open(json_file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print("Đã nạp thành công file JSON!")
print(f"Tổng số bức ảnh: {len(data['images'])}")
print(f"Tổng số biển báo (bounding boxes): {len(data['annotations'])}")

In [ ]:
annotations = data['annotations']
df_anno = pd.DataFrame(annotations)

class_counts = df_anno['category_id'].value_counts().reset_index()
class_counts.columns = ['Category ID', 'Count']

plt.figure(figsize=(12, 6))
sns.barplot(data=class_counts, x='Category ID', y='Count', palette='viridis', hue='Category ID', legend=False)
plt.title('Phân bố số lượng các loại Biển báo (Class Distribution)', fontsize=16)
plt.xlabel('Category ID (Loại biển báo)', fontsize=12)
plt.ylabel('Số lượng (Count)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
df_anno['bbox_width'] = df_anno['bbox'].apply(lambda x: x[2])
df_anno['bbox_height'] = df_anno['bbox'].apply(lambda x: x[3])
df_anno['bbox_area'] = df_anno['bbox_width'] * df_anno['bbox_height']

images = data['images']
df_img = pd.DataFrame(images)[['id', 'width', 'height']].rename(columns={'id': 'image_id', 'width': 'img_w', 'height': 'img_h'})

df_merged = df_anno.merge(df_img, on='image_id')
df_merged['img_area'] = df_merged['img_w'] * df_merged['img_h']
df_merged['area_ratio (%)'] = (df_merged['bbox_area'] / df_merged['img_area']) * 100

plt.figure(figsize=(10, 6))
sns.histplot(df_merged['area_ratio (%)'], bins=50, color='red', kde=True)
plt.title('Phân bố Kích thước Biển báo so với toàn bức ảnh', fontsize=16)
plt.xlabel('Tỷ lệ diện tích của Biển báo / Diện tích ảnh (%)', fontsize=12)
plt.ylabel('Tần suất (Số lượng biển báo)', fontsize=12)
plt.xlim(0, 5)
plt.grid(linestyle='--', alpha=0.7)

plt.savefig('bbox_size_distribution.png', bbox_inches='tight')
plt.show()

tiny_boxes_ratio = (len(df_merged[df_merged['area_ratio (%)'] < 1.0]) / len(df_merged)) * 100
print(f"-> LUẬN ĐIỂM BÁO CÁO: Có tới {tiny_boxes_ratio:.2f}% số biển báo có diện tích < 1% diện tích ảnh gốc.")